# Dinâmica da Criminalidade no Rio de Janeiro: Uma Análise Integrada Sobre Segurança Pública

*Análise Exploratória de Dados e Visualização*  
*FGV EMAp - Escola de Matemática Aplicada | Semestre 2026.2*

*Professor:* João Rulff  
 
*Autores:*   
Josué Santos de Oliveira da Silva   
Leonardo Cardia Bicca   
Luigi Parmeggiani Porto

Repositório do projeto: https://github.com/Josue-santoss/AEDV-A2-.git

## Introdução:

&emsp;O estado do Rio de Janeiro vivencia, há décadas, um dos cenários de segurança pública mais desafiadores e militarizados da América Latina. O presente artigo investiga a arquitetura espaço-temporal da violência letal fluminense a partir do cruzamento de 15 bases de microdados abertos fornecidas pelo Instituto de Segurança Pública (ISP-RJ), cobrindo, conforme a base, o período longitudinal de 2003 a 2025. Estruturado em torno de sete perguntas de domínio (este notebook desenvolve as Perguntas 1, 2, 3 e 5), o estudo aborda desde a substituição tecnológica nos arsenais do crime organizado até as dinâmicas territoriais de letalidade policial e violência de gênero.  

## O Que o Leitor deve Saber:

1. Sobre os dados: Os dados têm origem no Instituto de Segurança Pública do Estado do Rio de Janeiro (ISP-RJ) e estão
estruturados em um conjunto de bases de dados abertas de frequências mensal e anual. A coleta e agregação
originais são feitas através dos registros de ocorrência das delegacias do estado.
Utilizaremos diversas tabelas relacionais que cobrem diferentes dimensões da segurança. As variáveis
mais relevantes para a análise incluem os recortes temporais e geográficos, tipificações de armas apreendidas
(com destaque para fuzis) e contagens e taxas (por 100 mil habitantes) de crimes contra a vida. Utilizaremos
também a tabela de relação CISP-AISP-RISP para realizar cruzamentos de dados espaciais entre os níveis
de delegacia, município e estado.


2. Siglas:  
- CISP: A Circunscrição Integrada de Segurança Pública é a área territorial que coincide com a jurisdição de uma delegacia distrital da Polícia Civil (SEPOL) e a subárea correspondente de uma companhia da Polícia Militar (SEPM).  
- AISP: A Área Integrada de Segurança Pública é uma divisão territorial que alinha a atuação da Polícia Militar (PMERJ) e da Polícia Civil (PCERJ) na mesma região para combater a criminalidade de forma conjunta.  
- RISP: A Região Integrada de Segurança Pública une os comandos intermediários das polícias, ou seja, os Departamentos de Polícia de Área (DPA) da Polícia Civil e os Comandos de Policiamento de Área (CPA) da Polícia Militar.

3. Convenções e limitações adotadas neste notebook:
- **Períodos:** as bases cobrem 2003 em diante (algumas até 2025). Séries anuais usam apenas **anos com 12 meses** de dados; os anos incompletos são descartados e o notebook imprime quais são.
- **Taxas por 100 mil habitantes:** só existem onde há população. Para as AISPs, a população cobre apenas **2003–2022**; nenhum ano é projetado. A única exceção é a Pergunta 1 (população estadual), em que o último valor observado é repetido e os anos afetados são **destacados no gráfico**.
- **Feminicídio de 2016** foi excluído: o total do ano (14) é muito inferior ao de 2017 (68), padrão compatível com registro parcial no início da série. A célula de diagnóstico mostra os números para conferência.
- **AISPs 1 e 13** não têm população na base e, portanto, não têm taxas.
- **Correlações entre séries anuais** (poucos pontos, com tendência) são **descritivas**: sempre acompanhadas de Spearman e da correlação das variações anuais, nunca interpretadas como causais.

# Preparação das bases de dados

Esta parte do notebook constitui a **camada comum de dados** do projeto. Ela é executada antes das perguntas de pesquisa. Cada tabela derivada tem uma granularidade definida (por exemplo, uma linha por `ano` e `id_aisp`) e pode ser reutilizada por novas análises.

**Princípios:** preservar as bases originais em `dataframes_brutos`, manter as versões tratadas em `dataframes`, separar limpeza de relacionamento, construir indicadores uma única vez e registrar lacunas de cobertura sem inventar observações.

## 1. Bibliotecas e configuração

Todos os imports e **todos os parâmetros analíticos** ficam nesta célula, para que períodos, semente aleatória e cores sejam os mesmos em todo o notebook. Para mudar a fonte dos dados, basta alterar `REPOSITORIO_DADOS` (ou fixar `BRANCH_DADOS` em um *hash* de commit, o que garante reprodutibilidade total). O ZIP do repositório é guardado em `ARQUIVO_CACHE_ZIP`; apague o arquivo para forçar um novo download.

In [ ]:
import os

# Evita o aviso de vazamento de memória do KMeans (Windows + MKL).
# Precisa ser definido antes de importar numpy/sklearn.
os.environ.setdefault("OMP_NUM_THREADS", "1")

import tempfile
import warnings
from io import BytesIO
from zipfile import ZipFile

import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
import plotly.express as px
import requests
from IPython.display import display
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", message=".*KMeans is known to have a memory leak.*")

# --- Fonte dos dados ---
REPOSITORIO_DADOS = "josue-fgv/AEDV-A2-"
BRANCH_DADOS = "main"  # para reprodutibilidade total, troque por um hash de commit
PASTA_DADOS_ZIP = "/Projeto/Datas/"
PREFIXO_CSV = "br_rj_isp_estatisticas_seguranca_"
ARQUIVO_CACHE_ZIP = "cache_repositorio_isp.zip"  # apague para forçar novo download

URL_ZIP_DADOS = (
    f"https://github.com/{REPOSITORIO_DADOS}/archive/refs/heads/{BRANCH_DADOS}.zip"
)

# --- Parâmetros analíticos (centralizados para manter os períodos coerentes) ---
FATOR_TAXA = 100_000
SEMENTE = 42
ANO_INICIAL_ARMAS = 2007           # início da série de armas usada na Pergunta 1
ANOS_EXCLUIDOS_FEMINICIDIO = {2016}  # ano com registro parcial (ver diagnóstico na seção 16)
ANOS_NOS_MAPAS = 5                 # janela (em anos) dos mapas animados
TOP_N = 10                         # tamanho dos rankings
TOLERANCIA_SIMPLIFICACAO = 0.005   # em graus (~500 m); reduz o peso dos mapas

# Paleta segura para daltonismo (Okabe-Ito)
PALETA = {
    "azul": "#0072B2",
    "vermelho": "#D55E00",
    "verde": "#009E73",
    "laranja": "#E69F00",
    "amarelo": "#F0E442",
    "ciano": "#56B4E9",
    "cinza": "#7F7F7F",
}


## 2. Carregamento e catálogo das bases

O ZIP do repositório é baixado **uma única vez** (e reaproveitado do cache local nas execuções seguintes). Dele saem tanto os CSVs quanto os shapefiles, sem um segundo download. Os arquivos CSV são identificados automaticamente a partir da pasta de dados; o separador é detectado pelo cabeçalho, o que é muito mais rápido do que o *sniffer* do pandas (`sep=None, engine="python"`), mantido apenas como plano B.

In [ ]:
def baixar_repositorio(url_zip: str, arquivo_cache: str | None = None) -> bytes:
    """Baixa o ZIP do repositório ou reaproveita a cópia local."""
    if arquivo_cache and os.path.exists(arquivo_cache):
        with open(arquivo_cache, "rb") as arquivo:
            return arquivo.read()
    resposta = requests.get(url_zip, timeout=120)
    resposta.raise_for_status()
    if arquivo_cache:
        with open(arquivo_cache, "wb") as arquivo:
            arquivo.write(resposta.content)
    return resposta.content


def ler_csv_detectando_separador(conteudo: bytes) -> pd.DataFrame:
    """Detecta o separador pelo cabeçalho e lê com o motor rápido do pandas."""
    cabecalho = conteudo.split(b"\n", 1)[0].decode("utf-8-sig", errors="replace")
    separador = max([",", ";", "\t", "|"], key=cabecalho.count)
    base = pd.read_csv(BytesIO(conteudo), sep=separador, encoding="utf-8-sig")
    if base.shape[1] == 1:  # detecção falhou: usa o sniffer do pandas
        base = pd.read_csv(
            BytesIO(conteudo), sep=None, engine="python", encoding="utf-8-sig"
        )
    return base


def carregar_csvs_do_zip(zip_bytes: bytes, pasta_dados: str, prefixo: str) -> dict[str, pd.DataFrame]:
    """Lê os CSVs do ZIP; mantém cada tabela separada e identificada pelo nome."""
    bases_carregadas = {}
    with ZipFile(BytesIO(zip_bytes)) as arquivo_zip:
        for caminho in arquivo_zip.namelist():
            if pasta_dados not in caminho or not caminho.lower().endswith(".csv"):
                continue
            nome_arquivo = caminho.rsplit("/", 1)[-1]
            nome_base = nome_arquivo.removesuffix(".csv").removeprefix(prefixo)
            if nome_base in bases_carregadas:
                raise ValueError(f"Nome de base duplicado no ZIP: {nome_base}")
            bases_carregadas[nome_base] = ler_csv_detectando_separador(
                arquivo_zip.read(caminho)
            )
    if not bases_carregadas:
        raise ValueError(f"Nenhum arquivo CSV encontrado em {pasta_dados!r}.")
    return bases_carregadas


def carregar_shapefile_do_zip(zip_bytes: bytes, nome: str, pasta_dados: str) -> gpd.GeoDataFrame:
    """Lê um shapefile compactado (ex.: AISP.zip) de dentro do ZIP do repositório."""
    with ZipFile(BytesIO(zip_bytes)) as arquivo_zip:
        candidatos = [
            caminho for caminho in arquivo_zip.namelist()
            if pasta_dados in caminho
            and caminho.rsplit("/", 1)[-1].lower() == f"{nome.lower()}.zip"
        ]
        if not candidatos:
            raise FileNotFoundError(f"{nome}.zip não encontrado em {pasta_dados!r}.")
        conteudo = arquivo_zip.read(candidatos[0])

    with tempfile.NamedTemporaryFile(delete=False, suffix=".zip") as arquivo_tmp:
        arquivo_tmp.write(conteudo)
        caminho_tmp = arquivo_tmp.name
    try:
        return gpd.read_file(f"zip://{caminho_tmp}")
    finally:
        os.unlink(caminho_tmp)


zip_repositorio = baixar_repositorio(URL_ZIP_DADOS, ARQUIVO_CACHE_ZIP)
dataframes_brutos = carregar_csvs_do_zip(zip_repositorio, PASTA_DADOS_ZIP, PREFIXO_CSV)
print(f"{len(dataframes_brutos)} bases importadas:", ", ".join(sorted(dataframes_brutos)))


## 3. Limpeza e padronização

Esta rotina produz **cópias** das bases de entrada. `dataframes_brutos` preserva os CSVs importados; `dataframes` contém as bases tratadas. Corrige textos, preserva valores ausentes de chaves/população e define uma coluna temporal `data` nas tabelas mensais.

**Critério mantido do código original:** ausências em colunas numéricas de contagem iniciadas por `quantidade_` são preenchidas com zero. Essa hipótese deve ser confrontada com o dicionário oficial dos dados; não deve ser automaticamente aplicada às novas métricas.

In [ ]:
CORRECOES_REGIAO = {
    "Grande NiterÃƒÂƒÃ‚ÂƒÃƒÂ‚Ã‚Â³i": "Grande Niterói",
    "Grande Niteroi": "Grande Niterói",
}


def exigir_colunas(base: pd.DataFrame, colunas: list[str], nome_base: str) -> None:
    """Interrompe o processamento caso falte alguma coluna esperada."""
    faltantes = set(colunas).difference(base.columns)
    if faltantes:
        raise KeyError(f"{nome_base}: colunas ausentes: {sorted(faltantes)}")


def limpar_bases_isp(bases: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:
    """Padroniza as bases sem alterar os DataFrames de entrada."""
    bases_tratadas = {}

    for nome_base, base_original in bases.items():
        base = base_original.copy()
        base.columns = base.columns.str.strip()

        # Limpeza de textos sem transformar ausências (NaN) em strings 'nan'.
        for coluna in base.select_dtypes(include=["object", "string"]).columns:
            base[coluna] = base[coluna].map(
                lambda valor: valor.strip() if isinstance(valor, str) else valor
            )

        if "regiao" in base.columns:
            base["regiao"] = base["regiao"].replace(CORRECOES_REGIAO)

        # Hipótese de imputação do projeto: somente contagens de ocorrências.
        for coluna in base.select_dtypes(include="number").columns:
            if coluna.startswith("quantidade_"):
                base[coluna] = base[coluna].fillna(0)

        if "ano" in base.columns:
            base["ano"] = pd.to_numeric(base["ano"], errors="raise").astype("int64")
        if "mes" in base.columns:
            base["mes"] = pd.to_numeric(base["mes"], errors="raise").astype("int64")
        if {"ano", "mes"}.issubset(base.columns):
            base["data"] = pd.to_datetime(
                {"year": base["ano"], "month": base["mes"], "day": 1},
                errors="raise"
            )

        bases_tratadas[nome_base] = base

    return bases_tratadas


dataframes = limpar_bases_isp(dataframes_brutos)
print("Bases padronizadas:", len(dataframes))

## 4. Referências às bases de trabalho

Os apelidos abaixo conservam os nomes esperados pelas células das perguntas. **As bases continuam disponíveis em `dataframes`**; os apelidos não representam novas cópias.

In [ ]:
(
    df_armas,
    df_armas_fogo,
    df_cisp_mensal,
    df_mun_mensal,
    df_evol_uf,
    df_evol_upp,
    df_policial,
    df_fem_cisp,
    df_rel,
    df_taxa_anual_municipio,
    df_taxa_anual_uf,
    df_taxa_mun_mensal,
    df_taxa_mensal_uf,
    df_letalidade,
    df_populacao_mensal_cisp,
) = (
    dataframes["armas_apreendidas_mensal"],
    dataframes["armas_fogo_apreendidas_mensal"],
    dataframes["evolucao_mensal_cisp"],
    dataframes["evolucao_mensal_municipio"],
    dataframes["evolucao_mensal_uf"],
    dataframes["evolucao_mensal_upp"],
    dataframes["evolucao_policial_morto_servico_mensal"],
    dataframes["feminicidio_mensal_cisp"],
    dataframes["relacao_cisp_aisp_risp"],
    dataframes["taxa_evolucao_anual_municipio"],
    dataframes["taxa_evolucao_anual_uf"],
    dataframes["taxa_evolucao_mensal_municipio"],
    dataframes["taxa_evolucao_mensal_uf"],
    dataframes["taxa_letalidade"],
    dataframes["PopulacaoEvolucaoMensalCisp"],
)

# Tabela de inventário: facilita identificar novas fontes e colunas disponíveis.
inventario_bases = pd.DataFrame([
    {"base": nome_base, "linhas": len(base), "colunas": len(base.columns)}
    for nome_base, base in sorted(dataframes.items())
])
inventario_bases

# Relacionamentos e dimensões territoriais

As relações territoriais são construídas **antes** de qualquer cálculo de taxas. Uma chave de relacionamento não deve ser substituída por uma descrição textual. As descrições ficam em tabelas de dimensão.

## 5. Relações entre CISP, AISP e regiões

`df_rel` é a referência da correspondência CISP–AISP–RISP. O agrupamento por região pode conter uma AISP em mais de uma categoria; portanto, a relação `regiao → AISP` não é pressuposta como exclusiva.

In [ ]:
exigir_colunas(df_rel, ["id_cisp", "id_aisp", "regiao"], "df_rel")

# Estrutura consultável pelo nome da região (sem depender da ordem do groupby).
grupos = df_rel.groupby("regiao")["id_aisp"].unique()
aisps_por_regiao = {
    regiao: sorted(int(aisp) for aisp in identificadores if pd.notna(aisp))
    for regiao, identificadores in grupos.items()
}

print("Regiões disponíveis:", ", ".join(aisps_por_regiao))

## 6. Padronização da população mensal por CISP

**Granularidade:** `(ano, mes, id_cisp)`.

Conserva-se a metodologia do notebook original: converter a população mensal para inteiro, truncando a parte decimal, **antes** do cálculo da média anual. Não se soma a população dos meses.

In [ ]:
# Trabalhar em cópia para não alterar a tabela importada em dataframes.
df_populacao_mensal_cisp = df_populacao_mensal_cisp.rename(
    columns={"circ": "id_cisp", "pop_circ": "populacao"}
).copy()
exigir_colunas(
    df_populacao_mensal_cisp,
    ["ano", "mes", "id_cisp", "populacao"],
    "df_populacao_mensal_cisp",
)

populacao_texto = (
    df_populacao_mensal_cisp["populacao"]
    .astype("string")
    .str.replace(",", ".", regex=False)
)
populacao_numerica = pd.to_numeric(populacao_texto, errors="raise")
if populacao_numerica.isna().any() or (populacao_numerica < 0).any():
    raise ValueError("Há populações ausentes ou negativas na base mensal por CISP.")

df_populacao_mensal_cisp["populacao"] = populacao_numerica.astype("int64")

duplicatas_mensais = df_populacao_mensal_cisp.duplicated(
    subset=["ano", "mes", "id_cisp"], keep=False
)
if duplicatas_mensais.any():
    raise ValueError("Há registros duplicados de (ano, mes, id_cisp) na população mensal.")

## 7. População anual por CISP

**Granularidade:** `(ano, id_cisp)`. A estimativa anual é a média dos valores mensais disponíveis, como no cálculo original. Para futuras análises, a base mensal permanece acessível separadamente.

In [ ]:
df_populacao_anual_cisp = (
    df_populacao_mensal_cisp
    .groupby(["ano", "id_cisp"], as_index=False)["populacao"]
    .mean()
)
df_populacao_anual_cisp.head()

## 8. Dimensão de relacionamento CISP → AISP

Atribui-se uma AISP por CISP, como no notebook original. Diferentemente de `.first()` sem verificação, a tabela de correspondência explicita chaves e sinaliza conflitos. **Uma eventual mudança histórica de jurisdição exige uma tabela CISP–AISP com vigência temporal; o arquivo atual não é tratado como histórico.**

In [ ]:
df_mapa_cisp_aisp = (
    df_rel[["id_cisp", "id_aisp"]]
    .dropna(subset=["id_cisp", "id_aisp"])
    .drop_duplicates()
    .copy()
)

conflitos_cisp_aisp = (
    df_mapa_cisp_aisp
    .groupby("id_cisp")["id_aisp"]
    .nunique()
    .loc[lambda serie: serie > 1]
)
if not conflitos_cisp_aisp.empty:
    raise ValueError(
        "Uma CISP foi associada a mais de uma AISP. "
        "Não é seguro escolher .first(); verifique a correspondência temporal. "
        f"Exemplos: {conflitos_cisp_aisp.head().to_dict()}"
    )

# Mantido como recurso de consulta, sem ser a fonte primária da relação.
aisp_de_cisp = df_mapa_cisp_aisp.set_index("id_cisp")["id_aisp"].to_dict()

## 9. População anual por AISP

**Granularidade:** `(ano, id_aisp)`. Primeiro associamos a CISP à AISP (`many_to_one`); depois, somamos as estimativas anuais das CISPs pertencentes a cada AISP. CISPs sem correspondência são registradas em `populacao_cisp_sem_aisp`, em vez de serem ignoradas sem rastreabilidade.

In [ ]:
populacao_cisp_com_aisp = df_populacao_anual_cisp.merge(
    df_mapa_cisp_aisp,
    on="id_cisp",
    how="left",
    validate="many_to_one",
    indicator="situacao_mapeamento",
)

populacao_cisp_sem_aisp = populacao_cisp_com_aisp.loc[
    populacao_cisp_com_aisp["id_aisp"].isna(),
    ["ano", "id_cisp", "populacao"],
].copy()

populacao_por_ano_aisp = (
    populacao_cisp_com_aisp
    .dropna(subset=["id_aisp"])
    .groupby(["ano", "id_aisp"], as_index=False)["populacao"]
    .sum(min_count=1)
)

print("CISPs/anos sem AISP associada:", len(populacao_cisp_sem_aisp))
populacao_por_ano_aisp.head()

## 10. Dimensão de nomes das AISPs

Nomes de apresentação devem ter **uma fonte e uma definição únicas**. `dim_aisp` pode receber posteriormente colunas de localização, categorias e metadados; os indicadores continuam vinculados por `id_aisp`.

Referência declarada no notebook original: [Relação RISP × AISP × CISP — ISP-RJ](https://www.ispdados.rj.gov.br/Arquivos/Relacao_RISPxAISPxCISP.pdf). A nomenclatura abaixo foi preservada do trabalho, **sem validação histórica de vigência**.

In [ ]:
nomes_aisp = {
    1: "São Gonçalo",
    2: "Botafogo / Flamengo",
    3: "Méier / Cachambi",
    4: "São Cristóvão / Estácio",
    5: "Centro / Santa Teresa",
    6: "Tijuca / Maracanã",
    7: "São Gonçalo",
    8: "Campos dos Goytacazes",
    9: "Madureira / Cascadura",
    10: "Barra do Piraí / Valença",
    11: "Nova Friburgo",
    12: "Niterói / Maricá",
    14: "Bangu / Realengo",
    15: "Duque de Caxias",
    16: "Penha / Olaria",
    17: "Ilha do Governador",
    18: "Jacarepaguá",
    19: "Copacabana / Leme",
    20: "Nova Iguaçu / Mesquita",
    21: "São João de Meriti",
    22: "Maré / Bonsucesso",
    23: "Ipanema / Leblon / Rocinha",
    24: "Itaguaí / Seropédica",
    25: "Região dos Lagos",
    26: "Petrópolis",
    27: "Santa Cruz / Guaratiba",
    28: "Volta Redonda / Barra Mansa",
    29: "Itaperuna",
    30: "Teresópolis",
    31: "Barra da Tijuca / Recreio",
    32: "Macaé / Rio das Ostras",
    33: "Angra dos Reis / Mangaratiba",
    34: "Magé / Guapimirim",
    35: "Itaboraí / Rio Bonito",
    36: "Santo Antônio de Pádua",
    37: "Resende / Itatiaia",
    38: "Três Rios / Paraíba do Sul",
    39: "Belford Roxo",
    40: "Campo Grande",
    41: "Irajá / Pavuna",
    43: "Paraty",
}

# Tabela de dimensão para cruzamentos presentes e futuros.
dim_aisp = pd.DataFrame(
    [(codigo, descricao) for codigo, descricao in nomes_aisp.items()],
    columns=["id_aisp", "nome_aisp"],
)
if dim_aisp["id_aisp"].duplicated().any():
    raise ValueError("Há identificadores de AISP repetidos na dimensão.")

# Rótulo único de apresentação (usado em mapas, rankings e hovers de todo o notebook).
dim_aisp["rotulo_aisp"] = (
    "AISP " + dim_aisp["id_aisp"].astype(str).str.zfill(2) + " – " + dim_aisp["nome_aisp"]
)

# Validação: nomes repetidos entre AISPs diferentes costumam indicar erro de digitação.
nomes_repetidos = dim_aisp[dim_aisp["nome_aisp"].duplicated(keep=False)]
if not nomes_repetidos.empty:
    print("Atenção: nomes de AISP repetidos; confira a nomenclatura na fonte oficial:")
    display(nomes_repetidos)

populacao_por_ano_aisp = populacao_por_ano_aisp.merge(
    dim_aisp,
    on="id_aisp",
    how="left",
    validate="many_to_one",
)
populacao_por_ano_aisp.head()

# Construção de indicadores gerais

Estas tabelas são **insumos reutilizáveis**, não respostas a uma pergunta específica. Novos indicadores devem nascer a partir das mesmas chaves e das mesmas bases de referência, sem recalcular ou modificar as tabelas originais.

## 11. Contagens anuais de letalidade violenta por AISP

**Granularidade:** `(ano, id_aisp)`. Mantém-se a soma das contagens mensais. A função aceita outras colunas de contagem quando precisarmos ampliar os indicadores de segurança pública.

In [ ]:
def agregar_contagens_aisp_ano(
    base: pd.DataFrame,
    colunas_contagem: list[str],
) -> pd.DataFrame:
    """Agrupa contagens aditivas por ano e AISP."""
    exigir_colunas(base, ["ano", "id_aisp", *colunas_contagem], "base de ocorrências")
    return (
        base.groupby(["ano", "id_aisp"], as_index=False)[colunas_contagem]
        .sum()
    )


mortes_aisp_ano = agregar_contagens_aisp_ano(
    df_cisp_mensal,
    ["quantidade_letalidade_violenta"],
)

# Total anual estadual, sem período fixo: cada pergunta aplica seu recorte temporal.
letalidade_anual = (
    mortes_aisp_ano
    .groupby("ano", as_index=False)["quantidade_letalidade_violenta"]
    .sum()
    .sort_values("ano")
)
mortes_aisp_ano.head()

## 12. Painel AISP–ano e taxas por 100 mil habitantes

**Granularidade:** `(ano, id_aisp)`. `estatisticas_aisp` reúne contagem, população, nome e taxa; `validate='one_to_one'` verifica se os dois lados possuem chaves únicas. Não se calculam taxas para populações nulas ou não positivas: tais casos permanecem como `NaN` e são documentados no diagnóstico.

A série populacional fornecida cobre menos anos do que as ocorrências. O painel de taxas é restringido aos anos disponíveis na população; as contagens originais permanecem completas.

In [ ]:
def taxa_por_100_mil(
    quantidade: pd.Series,
    populacao: pd.Series,
    fator: int = FATOR_TAXA,
) -> pd.Series:
    """Taxa por habitante, com NaN quando falta denominador válido."""
    denominador_valido = populacao.where(populacao > 0)
    return quantidade.div(denominador_valido).mul(fator)


anos_com_populacao = populacao_por_ano_aisp["ano"].unique()

estatisticas_aisp = (
    mortes_aisp_ano.loc[mortes_aisp_ano["ano"].isin(anos_com_populacao)]
    .merge(
        populacao_por_ano_aisp,
        on=["ano", "id_aisp"],
        how="left",
        validate="one_to_one",
        indicator=True,
    )
)

estatisticas_aisp["taxa_letalidade"] = taxa_por_100_mil(
    estatisticas_aisp["quantidade_letalidade_violenta"],
    estatisticas_aisp["populacao"],
)
estatisticas_aisp.head()

## 13. Verificações da cobertura do painel

As AISPs sem população não devem receber taxas artificiais. Este diagnóstico pode ser estendido a novos cruzamentos e serve de referência para avaliar a cobertura das séries.

In [ ]:
diagnostico_integracao_aisp = (
    estatisticas_aisp["_merge"]
    .value_counts()
    .rename_axis("situacao")
    .reset_index(name="quantidade_linhas")
)

linhas_sem_taxa_letalidade = estatisticas_aisp.loc[
    estatisticas_aisp["taxa_letalidade"].isna(),
    ["ano", "id_aisp", "nome_aisp", "quantidade_letalidade_violenta", "populacao"],
].copy()

print("Cobertura do relacionamento mortes × população:")
print(diagnostico_integracao_aisp.to_string(index=False))
print("Linhas sem taxa de letalidade:", len(linhas_sem_taxa_letalidade))
print("CISPs/anos sem relacionamento territorial:", len(populacao_cisp_sem_aisp))
aisps_sem_nome = sorted(
    set(mortes_aisp_ano["id_aisp"].dropna().astype(int))
    - set(dim_aisp["id_aisp"])
)
if aisps_sem_nome:
    print("Identificadores de AISP sem nome na dimensão:", aisps_sem_nome)
if not linhas_sem_taxa_letalidade.empty:
    display(linhas_sem_taxa_letalidade.head(10))

# Estatísticas descritivas reutilizáveis

Este bloco sintetiza a **distribuição entre AISPs**, e não a taxa do estado. A média simples de taxas por AISP dá o mesmo peso a cada AISP; a taxa estadual exigiria somar os eventos e as populações antes da divisão.

## 14. Resumo anual das taxas por AISP

O cálculo do desvio-padrão mantém `ddof=0`. Para ampliar o projeto, a função pode receber outra coluna de taxa e retornar o mesmo conjunto de estatísticas por ano.

In [ ]:
def resumir_taxas_por_ano(
    painel: pd.DataFrame,
    coluna_taxa: str = "taxa_letalidade",
) -> pd.DataFrame:
    """Resume a distribuição anual de uma taxa entre AISPs."""
    exigir_colunas(painel, ["ano", coluna_taxa], "painel de taxas")
    resumo = (
        painel.groupby("ano")[coluna_taxa]
        .agg(
            media="mean",
            mediana="median",
            minimo="min",
            maximo="max",
            q1=lambda serie: serie.quantile(0.25),
            q3=lambda serie: serie.quantile(0.75),
            desvio_padrao=lambda serie: serie.std(ddof=0),
            n_aisp="count",
        )
        .reset_index()
    )
    resumo["variancia"] = resumo["desvio_padrao"] ** 2
    resumo["amplitude_interquartil"] = resumo["q3"] - resumo["q1"]
    resumo["coef_variacao"] = (
        resumo["desvio_padrao"]
        .div(resumo["media"].where(resumo["media"] != 0))
        .mul(100)
    )
    return resumo


resumo_estatistico = resumir_taxas_por_ano(estatisticas_aisp)
resumo_estatistico.head()

## 15. Contrato de dados para as próximas análises

| Objeto | Uma linha por | Uso previsto |
|:--|:--|:--|
| `dataframes_brutos` | depende da fonte | CSVs originais para auditoria e reprocessamento |
| `dataframes` | depende da fonte | dados tratados para diferentes perguntas |
| `df_rel` | conforme arquivo ISP | relações territoriais de origem |
| `df_mapa_cisp_aisp` | CISP | correspondência territorial sem ambiguidades |
| `dim_aisp` | AISP | atributos descritivos expansíveis |
| `df_populacao_mensal_cisp` | ano, mês e CISP | série populacional de origem |
| `df_populacao_anual_cisp` | ano e CISP | média das populações mensais |
| `populacao_por_ano_aisp` | ano e AISP | denominador anual para indicadores |
| `mortes_aisp_ano` | ano e AISP | contagem anual de letalidade violenta |
| `estatisticas_aisp` | ano e AISP | painel de métricas integradas |
| `resumo_estatistico` | ano | distribuição de taxas entre AISPs |
| `zip_repositorio` | repositório | ZIP baixado uma vez (CSVs e shapefiles), com cache local |
| `painel_aisp` | ano e AISP | letalidade, intervenção policial, fuzis e feminicídio, com taxas por 100 mil hab. |
| `gdf_aisp` | AISP | polígonos padronizados (WGS 84, simplificados) para todos os mapas |

**Para novas métricas:** acrescentar a agregação correspondente, integrar pelo mesmo par de chaves e calcular a nova taxa com `taxa_por_100_mil`. **Para novos relacionamentos:** criar uma dimensão/correspondência validada, sem reutilizar variáveis para diferentes sentidos. Mudanças históricas dos limites de AISP/CISP requerem uma tabela temporal validada antes de comparar territórios ao longo do tempo.

## 16. Painel integrado AISP–ano

**Granularidade:** `(ano, id_aisp)`. Reúne letalidade violenta, intervenção policial, fuzis apreendidos e feminicídio, com as respectivas taxas por 100 mil habitantes. É a **fonte única** das Perguntas 2 e 5 (antes, cada pergunta refazia suas próprias agregações dentro de uma função de preparação). As contagens são somadas por AISP; as taxas só existem nos anos com população. Feminicídios dos anos excluídos em `ANOS_EXCLUIDOS_FEMINICIDIO` ficam como `NaN`, e não como zero.

O bloco também imprime a **cobertura mensal** de cada base e o diagnóstico do feminicídio de 2016, para que as exclusões possam ser conferidas.

In [ ]:
def anos_com_12_meses(base: pd.DataFrame) -> list[int]:
    """Anos em que a base tem os 12 meses (evita tratar ano parcial como ano completo)."""
    meses_por_ano = base.groupby("ano")["mes"].nunique()
    return sorted(int(ano) for ano in meses_por_ano[meses_por_ano == 12].index)


def padronizar_id_aisp(base: pd.DataFrame) -> pd.DataFrame:
    """Remove linhas sem AISP e garante id_aisp inteiro (chave comum a todas as tabelas)."""
    base = base.dropna(subset=["id_aisp"]).copy()
    base["id_aisp"] = base["id_aisp"].astype(int)
    return base


def milhar(valor: float, casas: int = 0) -> str:
    """Formata números no padrão brasileiro (1.234,5)."""
    texto = f"{valor:,.{casas}f}"
    return texto.replace(",", "X").replace(".", ",").replace("X", ".")


# --- Cobertura mensal das bases (anos com menos de 12 meses não entram em séries anuais) ---
for nome, base in {
    "letalidade (CISP)": df_cisp_mensal,
    "armas apreendidas": df_armas,
    "feminicídio": df_fem_cisp,
}.items():
    meses = base.groupby("ano")["mes"].nunique()
    incompletos = {int(ano): int(n) for ano, n in meses[meses < 12].items()}
    print(
        f"{nome}: anos {int(meses.index.min())}–{int(meses.index.max())}; "
        f"anos com menos de 12 meses: {incompletos or 'nenhum'}"
    )

# --- Diagnóstico do feminicídio nos primeiros anos da série ---
cobertura_feminicidio = df_fem_cisp.groupby("ano").agg(
    meses=("mes", "nunique"), feminicidios=("quantidade_morte_feminicidio", "sum")
)
print("\nFeminicídio, primeiros anos da série (justifica a exclusão de 2016):")
display(cobertura_feminicidio.head(4))

# --- Agregações anuais por AISP ---
fuzis_aisp_ano = (
    padronizar_id_aisp(df_armas)
    .groupby(["ano", "id_aisp"], as_index=False)["quantidade_fuzil"]
    .sum()
)
intervencao_aisp_ano = agregar_contagens_aisp_ano(
    padronizar_id_aisp(df_cisp_mensal), ["quantidade_homicidio_intervencao_policial"]
)
feminicidio_aisp_ano = (
    padronizar_id_aisp(df_fem_cisp)
    .groupby(["ano", "id_aisp"], as_index=False)["quantidade_morte_feminicidio"]
    .sum()
)
feminicidio_aisp_ano = feminicidio_aisp_ano[
    ~feminicidio_aisp_ano["ano"].isin(ANOS_EXCLUIDOS_FEMINICIDIO)
]

# --- Painel integrado (left join: nenhuma AISP-ano do painel de taxas é descartada) ---
chaves = ["ano", "id_aisp"]
painel_aisp = (
    padronizar_id_aisp(estatisticas_aisp.drop(columns="_merge"))
    .merge(fuzis_aisp_ano, on=chaves, how="left", validate="one_to_one")
    .merge(intervencao_aisp_ano, on=chaves, how="left", validate="one_to_one")
    .merge(feminicidio_aisp_ano, on=chaves, how="left", validate="one_to_one")
)
painel_aisp["rotulo_aisp"] = painel_aisp["rotulo_aisp"].fillna(
    "AISP " + painel_aisp["id_aisp"].astype(str).str.zfill(2)
)
painel_aisp["taxa_fuzil_100k"] = taxa_por_100_mil(
    painel_aisp["quantidade_fuzil"], painel_aisp["populacao"]
)
painel_aisp["taxa_intervencao_100k"] = taxa_por_100_mil(
    painel_aisp["quantidade_homicidio_intervencao_policial"], painel_aisp["populacao"]
)
# Atenção: o denominador é a população total (a base não traz população feminina).
painel_aisp["taxa_feminicidio_100k"] = taxa_por_100_mil(
    painel_aisp["quantidade_morte_feminicidio"], painel_aisp["populacao"]
)

com_populacao = painel_aisp[painel_aisp["populacao"].notna()]
ANO_INICIAL_POPULACAO = int(com_populacao["ano"].min())
ANO_FINAL_POPULACAO = int(com_populacao["ano"].max())
print(
    f"\nPainel AISP–ano: {len(painel_aisp)} linhas. Taxas disponíveis de "
    f"{ANO_INICIAL_POPULACAO} a {ANO_FINAL_POPULACAO}; "
    f"AISPs sem população (sem taxa): {sorted(int(i) for i in painel_aisp.loc[painel_aisp['populacao'].isna(), 'id_aisp'].unique())}"
)
print(
    "Linhas sem fuzis registrados:", int(painel_aisp["quantidade_fuzil"].isna().sum()),
    "| sem intervenção policial:", int(painel_aisp["quantidade_homicidio_intervencao_policial"].isna().sum()),
)
painel_aisp.head()


## 17. Geometrias das AISPs

O shapefile de AISPs sai do mesmo ZIP do repositório. A função padroniza a coluna de identificação (`id_aisp`, inteiro), converte para WGS 84, **dissolve** eventuais polígonos repetidos por AISP e **simplifica** a malha com `TOLERANCIA_SIMPLIFICACAO`. A simplificação é feita uma única vez, para que todos os mapas tenham o mesmo peso (antes, um dos mapas era gerado sem simplificação e produzia ~30 MB de saída). Os shapefiles de CISP e RISP, que não eram usados em nenhuma análise, deixaram de ser baixados.

In [ ]:
def preparar_geometria_aisp(gdf: gpd.GeoDataFrame, tolerancia: float) -> gpd.GeoDataFrame:
    """Padroniza chave, CRS e geometria das AISPs para uso em todos os mapas."""
    gdf = gdf.copy()
    coluna_id = next((c for c in gdf.columns if c.lower() in {"id_aisp", "aisp"}), None)
    if coluna_id is None:
        raise KeyError(f"Shapefile sem coluna de AISP. Colunas: {list(gdf.columns)}")
    gdf = gdf.rename(columns={coluna_id: "id_aisp"})
    gdf["id_aisp"] = pd.to_numeric(gdf["id_aisp"], errors="raise").astype(int)
    if gdf.crs is None:
        raise ValueError("O GeoDataFrame de AISPs não possui CRS definido.")
    gdf = gdf.to_crs(epsg=4326)
    gdf = gdf[["id_aisp", "geometry"]].dissolve(by="id_aisp", as_index=False)
    gdf["geometry"] = gdf.geometry.simplify(tolerancia, preserve_topology=True)
    return gdf


gdf_aisp = preparar_geometria_aisp(
    carregar_shapefile_do_zip(zip_repositorio, "AISP", PASTA_DADOS_ZIP),
    TOLERANCIA_SIMPLIFICACAO,
)

ids_painel = set(painel_aisp["id_aisp"])
ids_mapa = set(gdf_aisp["id_aisp"])
print(f"{len(gdf_aisp)} polígonos de AISP carregados.")
print("AISPs do painel sem polígono (não aparecem nos mapas):", sorted(int(i) for i in ids_painel - ids_mapa))
print("Polígonos sem dados no painel:", sorted(int(i) for i in ids_mapa - ids_painel))


# Perguntas:

## Pergunta 1 - Qual é a evolução histórica do volume absoluto e da taxa (por 100 mil habitantes) de armas de fogo e fuzis apreendidos no estado do Rio de Janeiro nos últimos anos?


In [ ]:
colunas_armas = [
    "quantidade_arma_fogo_total", "quantidade_fuzil",
    "quantidade_pistola", "quantidade_revolver",
]
exigir_colunas(df_armas, ["ano", "mes", *colunas_armas], "df_armas")

# Apenas anos completos, a partir de ANO_INICIAL_ARMAS.
anos_armas = [ano for ano in anos_com_12_meses(df_armas) if ano >= ANO_INICIAL_ARMAS]
armas_anual = (
    df_armas[df_armas["ano"].isin(anos_armas)]
    .groupby("ano")[colunas_armas]
    .sum()
)

# População estadual (a limpeza de "regiao" já foi feita na seção 3).
pop_uf = (
    df_letalidade.loc[df_letalidade["regiao"] == "Estado"]
    .groupby("ano")["populacao"]
    .first()
)
armas_anual["populacao"] = pop_uf.reindex(armas_anual.index)
armas_anual["populacao_estimada"] = armas_anual["populacao"].isna()
armas_anual["populacao"] = armas_anual["populacao"].ffill()  # última população observada
anos_pop_estimada = armas_anual.index[armas_anual["populacao_estimada"]].tolist()
if anos_pop_estimada:
    print(
        f"Atenção: sem população estadual em {anos_pop_estimada}; "
        "foi repetido o último valor observado (anos sombreados nos gráficos de taxa)."
    )

armas_anual["taxa_armas_100k"] = taxa_por_100_mil(
    armas_anual["quantidade_arma_fogo_total"], armas_anual["populacao"]
)
armas_anual["taxa_fuzil_100k"] = taxa_por_100_mil(
    armas_anual["quantidade_fuzil"], armas_anual["populacao"]
)
total_armas = armas_anual["quantidade_arma_fogo_total"].where(lambda s: s > 0)
for tipo in ["fuzil", "pistola", "revolver"]:
    armas_anual[f"prop_{tipo}_pct"] = armas_anual[f"quantidade_{tipo}"].div(total_armas).mul(100)
armas_anual["prop_outras_pct"] = 100 - armas_anual[
    ["prop_fuzil_pct", "prop_pistola_pct", "prop_revolver_pct"]
].sum(axis=1)


def desenhar_serie(ax, serie, cor, titulo, rotulo_y, marcador="o", anos_estimados=()):
    """Série anual com eixo y iniciando em zero (sem limites fixos) e anos estimados sombreados."""
    ax.plot(serie.index, serie.values, marker=marcador, color=cor, linewidth=2.2)
    if len(anos_estimados):
        ax.axvspan(min(anos_estimados) - 0.5, max(anos_estimados) + 0.5, color="0.92", zorder=0)
    ax.set_title(titulo, loc="left", fontsize=12)
    ax.set_ylabel(rotulo_y)
    ax.set_ylim(bottom=0)
    ax.grid(alpha=0.3)


anos_x = armas_anual.index
fig, eixos = plt.subplots(2, 2, figsize=(15, 9), sharex=True)
desenhar_serie(eixos[0, 0], armas_anual["quantidade_arma_fogo_total"], PALETA["azul"],
               "(A) Armas de fogo apreendidas (volume)", "Armas apreendidas")
desenhar_serie(eixos[0, 1], armas_anual["quantidade_fuzil"], PALETA["vermelho"],
               "(B) Fuzis apreendidos (volume)", "Fuzis apreendidos", marcador="s")
desenhar_serie(eixos[1, 0], armas_anual["taxa_armas_100k"], PALETA["azul"],
               "(C) Armas apreendidas por 100 mil hab.", "Taxa por 100 mil hab.",
               anos_estimados=anos_pop_estimada)
desenhar_serie(eixos[1, 1], armas_anual["taxa_fuzil_100k"], PALETA["vermelho"],
               "(D) Fuzis apreendidos por 100 mil hab.", "Taxa por 100 mil hab.",
               marcador="s", anos_estimados=anos_pop_estimada)
for ax in eixos[1]:
    ax.set_xticks(list(anos_x[::2]))
    ax.tick_params(axis="x", rotation=45)
nota = " Faixa cinza: população estadual repetida do último ano observado." if anos_pop_estimada else ""
fig.suptitle(
    f"Pergunta 1: armas de fogo e fuzis apreendidos no RJ ({int(anos_x.min())}–{int(anos_x.max())})",
    fontsize=15,
)
fig.text(0.5, 0.005, "Taxa = apreensões ÷ população × 100 mil; mede a ação policial, não o estoque de armas em circulação." + nota,
         ha="center", fontsize=9, color="0.35")
plt.tight_layout(rect=(0, 0.02, 1, 0.97))
plt.show()

# Composição do arsenal apreendido (soma 100%).
fig, ax = plt.subplots(figsize=(11, 5.5))
for coluna, rotulo, cor, marcador in [
    ("prop_revolver_pct", "Revólveres", PALETA["laranja"], "^"),
    ("prop_pistola_pct", "Pistolas", PALETA["verde"], "o"),
    ("prop_fuzil_pct", "Fuzis", PALETA["vermelho"], "s"),
    ("prop_outras_pct", "Outras", PALETA["cinza"], "d"),
]:
    ax.plot(anos_x, armas_anual[coluna], marker=marcador, color=cor, linewidth=2.2, label=rotulo)
ax.set_ylabel("Participação no total apreendido (%)")
ax.set_ylim(bottom=0)
ax.set_xticks(list(anos_x[::2]))
ax.grid(alpha=0.3)
ax.legend(frameon=True, loc="upper right")
ax.set_title("(E) Composição do arsenal apreendido", loc="left", fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# Leitura dos resultados (gerada a partir dos dados; não depende de números digitados à mão)
ano_ini, ano_fim = int(armas_anual.index.min()), int(armas_anual.index.max())
print(f"Período analisado: {ano_ini}–{ano_fim}\n")
for rotulo, coluna in [("Armas de fogo", "quantidade_arma_fogo_total"), ("Fuzis", "quantidade_fuzil")]:
    serie = armas_anual[coluna]
    ano_pico, valor_pico = int(serie.idxmax()), serie.max()
    print(
        f"{rotulo}: {milhar(serie.loc[ano_ini])} em {ano_ini}; pico de {milhar(valor_pico)} em {ano_pico}; "
        f"{milhar(serie.loc[ano_fim])} em {ano_fim} ({(serie.loc[ano_fim] / valor_pico - 1) * 100:+.1f}% em relação ao pico)."
    )
print(
    f"Participação dos fuzis no total apreendido: {armas_anual.loc[ano_ini, 'prop_fuzil_pct']:.1f}% em {ano_ini} "
    f"→ {armas_anual.loc[ano_fim, 'prop_fuzil_pct']:.1f}% em {ano_fim}."
)


**Como ler os gráficos da Pergunta 1.** Apreensões dependem do esforço e da estratégia policial: uma queda no volume ou na taxa não significa, necessariamente, menos armas em circulação. A participação dos fuzis (gráfico E) mostra a **mudança de composição** do que é apreendido, que é a evidência mais direta de "substituição tecnológica", mas ainda assim reflete o que a polícia consegue localizar. Cada indicador tem o seu próprio eixo (sem eixos duplos) para que as escalas não sugiram relações inexistentes.

## Pergunta 2 - Quais Áreas Integradas (AISP) concentram as maiores taxas proporcionais de letalidade violenta e feminicídio ao longo do tempo?

### 2.1 Mapas animados das taxas por 100 mil habitantes

A pergunta fala em taxas **proporcionais**; por isso os mapas usam taxas (e não contagens, que tornam as AISPs mais populosas sempre as mais escuras). As contagens absolutas continuam no *hover*. A escala de cores vai de zero ao **percentil 98** de todos os anos, para que poucos valores extremos não "apaguem" as diferenças entre as demais AISPs; valores acima do teto aparecem na cor máxima. Como a população só existe até `ANO_FINAL_POPULACAO`, a janela dos mapas termina nesse ano.

In [ ]:
def criar_mapa_animado(painel, gdf, coluna, titulo, rotulo_cor, anos, hover_extra):
    """Mapa coroplético animado por ano para uma taxa do painel AISP–ano."""
    dados = (
        painel[painel["ano"].isin(anos)]
        .dropna(subset=[coluna])
        .sort_values(["ano", "id_aisp"])
    )
    if dados.empty:
        raise ValueError(f"Sem dados de {coluna!r} nos anos {list(anos)}.")
    geometrias = gdf[gdf["id_aisp"].isin(dados["id_aisp"].unique())].set_index("id_aisp").geometry
    teto = float(dados[coluna].quantile(0.98))

    fig = px.choropleth_map(
        dados,
        geojson=geometrias,
        locations="id_aisp",
        color=coluna,
        color_continuous_scale="Reds",
        range_color=[0, teto],
        animation_frame="ano",
        hover_name="rotulo_aisp",
        hover_data={"id_aisp": False, coluna: ":.1f", **hover_extra},
        labels={coluna: rotulo_cor},
        center={"lat": -22.55, "lon": -43.15},
        zoom=7.4,
        map_style="carto-positron",
        title=titulo,
    )
    fig.update_traces(marker_opacity=0.85, marker_line_width=0.4, marker_line_color="black")
    fig.update_layout(height=650, margin=dict(l=20, r=20, t=80, b=20))
    return fig


ANOS_MAPA = list(range(ANO_FINAL_POPULACAO - ANOS_NOS_MAPAS + 1, ANO_FINAL_POPULACAO + 1))

fig_letalidade = criar_mapa_animado(
    painel_aisp, gdf_aisp, "taxa_letalidade",
    f"<b>Letalidade violenta por 100 mil habitantes, por AISP ({ANOS_MAPA[0]}–{ANOS_MAPA[-1]})</b>",
    "Taxa por 100 mil hab.", ANOS_MAPA,
    {"quantidade_letalidade_violenta": True, "populacao": ":,.0f"},
)
fig_letalidade.show()

fig_feminicidio = criar_mapa_animado(
    painel_aisp, gdf_aisp, "taxa_feminicidio_100k",
    f"<b>Feminicídios por 100 mil habitantes, por AISP ({ANOS_MAPA[0]}–{ANOS_MAPA[-1]})</b>",
    "Taxa por 100 mil hab. (população total)", ANOS_MAPA,
    {"quantidade_morte_feminicidio": True, "populacao": ":,.0f"},
)
fig_feminicidio.show()


### 2.2 Quais AISPs concentram as maiores taxas ao longo do tempo?

À esquerda, as `TOP_N` AISPs com maior **taxa média** no período; à direita, o mapa de calor das 15 AISPs com as maiores médias, ano a ano. Taxas de AISPs pequenas oscilam bastante de um ano para o outro (poucos casos sobre população pequena); por isso a média do período é mais confiável do que a leitura de um único ano, especialmente no caso do feminicídio.

In [ ]:
def ranking_e_heatmap(painel, coluna, titulo, rotulo_taxa, anos, cor_barras, cmap="Reds", k_heat=15):
    """Ranking das maiores taxas médias + mapa de calor AISP × ano."""
    base = painel[painel["ano"].isin(anos)].dropna(subset=[coluna])
    medias = (
        base.groupby(["id_aisp", "rotulo_aisp"], as_index=False)[coluna]
        .mean()
        .sort_values(coluna, ascending=False)
    )
    topo = medias.head(TOP_N).iloc[::-1]
    ids_heat = medias.head(k_heat)["id_aisp"].tolist()
    rotulos = medias.set_index("id_aisp")["rotulo_aisp"]
    matriz = base[base["id_aisp"].isin(ids_heat)].pivot(index="id_aisp", columns="ano", values=coluna).loc[ids_heat]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(17, 6), gridspec_kw={"width_ratios": [1, 1.3]})
    ax1.barh(topo["rotulo_aisp"], topo[coluna], color=cor_barras)
    ax1.set_xlabel(rotulo_taxa)
    ax1.set_title(f"Maiores taxas médias ({int(base['ano'].min())}–{int(base['ano'].max())})", loc="left")
    ax1.grid(axis="x", alpha=0.3)

    imagem = ax2.imshow(matriz.values, aspect="auto", cmap=cmap)
    ax2.set_xticks(range(len(matriz.columns)))
    ax2.set_xticklabels([int(a) for a in matriz.columns], rotation=45)
    ax2.set_yticks(range(len(matriz.index)))
    ax2.set_yticklabels([rotulos[i] for i in matriz.index], fontsize=9)
    ax2.set_title(f"Evolução anual das {k_heat} AISPs de maior média", loc="left")
    fig.colorbar(imagem, ax=ax2, label=rotulo_taxa)

    fig.suptitle(titulo, fontsize=14)
    plt.tight_layout()
    plt.show()
    return medias


anos_com_taxa = sorted(painel_aisp.loc[painel_aisp["populacao"].notna(), "ano"].unique())
ranking_letalidade = ranking_e_heatmap(
    painel_aisp, "taxa_letalidade", "Letalidade violenta: AISPs com as maiores taxas",
    "Taxa por 100 mil hab.", anos_com_taxa, PALETA["vermelho"],
)
ranking_feminicidio = ranking_e_heatmap(
    painel_aisp, "taxa_feminicidio_100k", "Feminicídio: AISPs com as maiores taxas",
    "Taxa por 100 mil hab. (população total)", anos_com_taxa, PALETA["azul"], cmap="Blues",
)

print("Maiores taxas médias de letalidade violenta:")
for _, linha in ranking_letalidade.head(3).iterrows():
    print(f"  {linha['rotulo_aisp']}: {linha['taxa_letalidade']:.1f} por 100 mil")
print("Maiores taxas médias de feminicídio:")
for _, linha in ranking_feminicidio.head(3).iterrows():
    print(f"  {linha['rotulo_aisp']}: {linha['taxa_feminicidio_100k']:.2f} por 100 mil")


### 2.3 A letalidade violenta ficou mais concentrada ou mais dispersa entre as AISPs?

O coeficiente de variação (CV) das taxas entre AISPs mede a **dispersão territorial**: quanto maior, mais desigual a distribuição da letalidade. Os gráficos abaixo mostram o CV, o total estadual de vítimas e o **número de AISPs com taxa calculada em cada ano**: se esse número muda, a comparação do CV entre anos também muda.

In [ ]:
cv_anual = resumo_estatistico.sort_values("ano").copy()
total_letalidade_ano = (
    mortes_aisp_ano.groupby("ano", as_index=False)["quantidade_letalidade_violenta"].sum()
)
correlacao_cv_letalidade = cv_anual[["ano", "coef_variacao", "n_aisp"]].merge(
    total_letalidade_ano, on="ano", validate="one_to_one"
)

ano_i, ano_f = int(cv_anual["ano"].min()), int(cv_anual["ano"].max())
fig, eixos = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
series = [
    ("coef_variacao", PALETA["azul"], "Coeficiente de variação (%)",
     "Dispersão da taxa de letalidade violenta entre AISPs"),
    ("quantidade_letalidade_violenta", PALETA["vermelho"], "Vítimas",
     "Total estadual de vítimas de letalidade violenta"),
    ("n_aisp", PALETA["cinza"], "Nº de AISPs",
     "AISPs com taxa calculada (denominador do CV)"),
]
for ax, (coluna, cor, rotulo_y, titulo) in zip(eixos, series):
    ax.plot(correlacao_cv_letalidade["ano"], correlacao_cv_letalidade[coluna],
            marker="o", color=cor, linewidth=2)
    ax.set_ylabel(rotulo_y)
    ax.set_title(titulo, loc="left", fontsize=11)
    ax.grid(alpha=0.3)
    ax.set_ylim(bottom=0)
eixos[-1].set_xticks(range(ano_i, ano_f + 1, 2))
eixos[-1].tick_params(axis="x", rotation=45)
eixos[-1].set_xlabel("Ano")
plt.tight_layout()
plt.show()


In [ ]:
def resumir_associacao(x: pd.Series, y: pd.Series) -> pd.DataFrame:
    """Pearson, Spearman e correlação das variações anuais (remove parte da tendência comum)."""
    r, p_r = stats.pearsonr(x, y)
    rho, p_rho = stats.spearmanr(x, y)
    r_dif, p_dif = stats.pearsonr(x.diff().dropna(), y.diff().dropna())
    return pd.DataFrame(
        {
            "medida": ["Pearson (níveis)", "Spearman (níveis)", "Pearson (variações anuais)"],
            "coeficiente": [r, rho, r_dif],
            "p_valor": [p_r, p_rho, p_dif],
            "n": [len(x), len(x), len(x) - 1],
        }
    ).round(4)


def tendencia(anos: pd.Series, valores: pd.Series) -> str:
    """Sentido da tendência linear de uma série anual."""
    return "alta" if np.polyfit(anos, valores, 1)[0] > 0 else "queda"


def plotar_dispersao_anos(dados, col_x, col_y, rotulo_x, rotulo_y, titulo):
    """Dispersão com um ponto por ano (rotulado) e reta de regressão."""
    x, y = dados[col_x], dados[col_y]
    a, b = np.polyfit(x, y, 1)
    xs = np.linspace(x.min(), x.max(), 100)
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.scatter(x, y, s=70, color=PALETA["azul"], zorder=3)
    ax.plot(xs, a * xs + b, color=PALETA["vermelho"], linewidth=2, label="Reta de mínimos quadrados")
    for _, linha in dados.iterrows():
        ax.annotate(str(int(linha["ano"])), (linha[col_x], linha[col_y]),
                    xytext=(5, 5), textcoords="offset points", fontsize=9)
    ax.set_xlabel(rotulo_x)
    ax.set_ylabel(rotulo_y)
    ax.set_title(titulo, loc="left")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()


tabela_cv = resumir_associacao(
    correlacao_cv_letalidade["quantidade_letalidade_violenta"], correlacao_cv_letalidade["coef_variacao"]
)
display(tabela_cv)
plotar_dispersao_anos(
    correlacao_cv_letalidade, "quantidade_letalidade_violenta", "coef_variacao",
    "Total anual de vítimas de letalidade violenta", "Coeficiente de variação (%)",
    "Letalidade violenta × dispersão territorial (um ponto por ano)",
)
r_niveis = tabela_cv.loc[0, "coeficiente"]
r_difs = tabela_cv.loc[2, "coeficiente"]
t_cv = tendencia(correlacao_cv_letalidade["ano"], correlacao_cv_letalidade["coef_variacao"])
t_total = tendencia(correlacao_cv_letalidade["ano"], correlacao_cv_letalidade["quantidade_letalidade_violenta"])
print(
    f"Nos níveis, r = {r_niveis:.2f}; nas variações anuais, r = {r_difs:.2f}. "
    f"No período, o CV tem tendência de {t_cv} e o total de vítimas, de {t_total}. "
    "Quando as duas séries têm tendência, a correlação nos níveis é inflada por ela: "
    "a medida mais informativa sobre se os movimentos andam juntos de um ano para o outro é a das variações."
)


### 2.4 Feminicídios por AISP e relação com a letalidade violenta

Os mapas de taxa estão na seção 2.1 e o ranking na 2.2. Aqui comparamos as séries **estaduais**. Duas observações importantes: (i) feminicídio é um subconjunto da letalidade violenta, então existe sobreposição mecânica entre as séries; (ii) o ano de 2016 foi excluído por registro parcial (ver seção 16), pois um ponto isolado e muito baixo (14 casos contra 68 em 2017) puxava a correlação para valores artificialmente fortes. Só entram anos com 12 meses nas duas bases.

In [ ]:
anos_fem = sorted(
    set(anos_com_12_meses(df_fem_cisp)) & (set(anos_com_12_meses(df_cisp_mensal)) - ANOS_EXCLUIDOS_FEMINICIDIO)
)
feminicidios_ano = (
    df_fem_cisp[df_fem_cisp["ano"].isin(anos_fem)]
    .groupby("ano", as_index=False)["quantidade_morte_feminicidio"].sum()
)
correlacao_feminicidio = feminicidios_ano.merge(
    total_letalidade_ano, on="ano", validate="one_to_one"
).sort_values("ano")
correlacao_feminicidio["pct_feminicidio_na_letalidade"] = (
    correlacao_feminicidio["quantidade_morte_feminicidio"]
    / correlacao_feminicidio["quantidade_letalidade_violenta"] * 100
)
display(correlacao_feminicidio)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.bar(correlacao_feminicidio["ano"], correlacao_feminicidio["quantidade_morte_feminicidio"], color=PALETA["azul"])
ax1.set_title("Mortes por feminicídio (RJ)", loc="left")
ax1.set_ylabel("Vítimas")
ax2.plot(correlacao_feminicidio["ano"], correlacao_feminicidio["pct_feminicidio_na_letalidade"],
         marker="o", color=PALETA["vermelho"], linewidth=2)
ax2.set_title("Feminicídios como % da letalidade violenta", loc="left")
ax2.set_ylabel("%")
ax2.set_ylim(bottom=0)
for ax in (ax1, ax2):
    ax.set_xticks(correlacao_feminicidio["ano"])
    ax.tick_params(axis="x", rotation=45)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

tabela_fem = resumir_associacao(
    correlacao_feminicidio["quantidade_letalidade_violenta"], correlacao_feminicidio["quantidade_morte_feminicidio"]
)
display(tabela_fem)
plotar_dispersao_anos(
    correlacao_feminicidio, "quantidade_letalidade_violenta", "quantidade_morte_feminicidio",
    "Total anual de vítimas de letalidade violenta", "Total anual de vítimas de feminicídio",
    "Feminicídios × letalidade violenta (um ponto por ano)",
)
print(
    f"Pearson nos níveis: {tabela_fem.loc[0, 'coeficiente']:.2f}; nas variações anuais: {tabela_fem.loc[2, 'coeficiente']:.2f} "
    f"(n = {len(correlacao_feminicidio)} anos, {anos_fem[0]}–{anos_fem[-1]}). "
    f"Tendências no período: letalidade em {tendencia(correlacao_feminicidio['ano'], correlacao_feminicidio['quantidade_letalidade_violenta'])}, "
    f"feminicídio em {tendencia(correlacao_feminicidio['ano'], correlacao_feminicidio['quantidade_morte_feminicidio'])}. "
    "Com poucos pontos e séries em tendência (e mudanças de registro e tipificação ao longo do tempo), "
    "a correlação nos níveis reflete sobretudo as tendências, e não uma relação entre os fenômenos."
)


## Pergunta 3

**Escopo:** existe um padrão sazonal mensal na letalidade violenta do estado e ele difere entre as regiões integradas (RISPs)?

**Método.** Para cada região e ano, a contagem mensal é convertida em **vítimas por dia** (para que fevereiro não pareça "mais seguro") e dividida pela média diária desse mesmo ano. Assim, cada mês vira um **índice sazonal** (100 = média do ano), que remove o nível de cada região e a tendência de longo prazo. O gráfico mostra a média desse índice nos anos completos. Os nomes das regiões vêm dos próprios dados (`regiao` predominante em cada `id_risp`), e não de um dicionário digitado à mão.

In [ ]:
exigir_colunas(
    df_cisp_mensal,
    ["ano", "mes", "id_risp", "regiao", "quantidade_letalidade_violenta"],
    "df_cisp_mensal",
)
base_sazonal = df_cisp_mensal[df_cisp_mensal["ano"].isin(anos_com_12_meses(df_cisp_mensal))].copy()
base_sazonal = base_sazonal.dropna(subset=["id_risp"])
base_sazonal["id_risp"] = base_sazonal["id_risp"].astype(int)

# Nome da região derivado dos dados, com checagem de consistência.
regiao_por_risp = base_sazonal.groupby("id_risp")["regiao"].agg(
    regiao_predominante=lambda s: s.mode().iat[0] if not s.mode().empty else "Sem região",
    n_regioes_distintas="nunique",
)
print("Correspondência RISP → região (confira se há RISPs com mais de uma região):")
display(regiao_por_risp)
rotulo_risp = {
    id_risp: f"RISP {id_risp} – {linha['regiao_predominante']}"
    for id_risp, linha in regiao_por_risp.iterrows()
}


def indice_sazonal_por_ano(base: pd.DataFrame, chaves: list[str]) -> pd.DataFrame:
    """Índice mensal (média diária do ano = 100), calculado ano a ano."""
    mensal = base.groupby([*chaves, "ano", "mes"], as_index=False)["quantidade_letalidade_violenta"].sum()
    dias = pd.to_datetime({"year": mensal["ano"], "month": mensal["mes"], "day": 1}).dt.days_in_month
    mensal["por_dia"] = mensal["quantidade_letalidade_violenta"] / dias
    media_ano = mensal.groupby([*chaves, "ano"])["por_dia"].transform("mean")
    mensal["indice"] = mensal["por_dia"] / media_ano.where(media_ano > 0) * 100
    return mensal


indice_estado_ano = indice_sazonal_por_ano(base_sazonal, [])
indice_risp_ano = indice_sazonal_por_ano(base_sazonal, ["id_risp"])

df_sazonal = pd.concat(
    [
        indice_risp_ano.groupby(["id_risp", "mes"], as_index=False)["indice"].mean()
        .assign(serie=lambda d: d["id_risp"].map(rotulo_risp)),
        indice_estado_ano.groupby("mes", as_index=False)["indice"].mean()
        .assign(serie="Estado (total)"),
    ],
    ignore_index=True,
)

meses = ["Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"]
anos_sazonal = sorted(base_sazonal["ano"].unique())

fig = px.line(
    df_sazonal, x="mes", y="indice", color="serie",
    title=f"Índice sazonal da letalidade violenta (média diária do ano = 100), {int(anos_sazonal[0])}–{int(anos_sazonal[-1])}",
    labels={"serie": "", "indice": "Índice sazonal", "mes": ""},
)
fig.update_layout(
    xaxis=dict(tickmode="array", tickvals=list(range(1, 13)), ticktext=meses),
    title_x=0.5, title_font_size=18, legend=dict(font=dict(size=13)),
)
fig.update_traces(line=dict(width=4, color="black"), opacity=0.7, selector=dict(name="Estado (total)"))
fig.add_hline(y=100, line_dash="dot", line_color="gray")
fig.show()

# Leitura quantitativa para o Estado: o padrão mensal se repete de ano para ano?
estado_mes = df_sazonal[df_sazonal["serie"] == "Estado (total)"].set_index("mes")["indice"]
matriz_anos = indice_estado_ano.pivot(index="ano", columns="mes", values="indice")
estatistica_friedman, p_friedman = stats.friedmanchisquare(*[matriz_anos[m] for m in range(1, 13)])
print(
    f"Estado: maior índice em {meses[int(estado_mes.idxmax()) - 1]} ({estado_mes.max():.1f}), "
    f"menor em {meses[int(estado_mes.idxmin()) - 1]} ({estado_mes.min():.1f}); "
    f"amplitude de {estado_mes.max() - estado_mes.min():.1f} pontos."
)
print(
    f"Teste de Friedman (os meses diferem de modo consistente entre {matriz_anos.shape[0]} anos?): "
    f"estatística = {estatistica_friedman:.1f}, p = {p_friedman:.4f}."
)


## Pergunta 5

**Escopo:** territórios com maior apreensão de fuzis coexistem com maiores taxas de mortes por intervenção policial?

**Método.** Para o ano mais recente com população disponível, cada AISP é descrita por duas **taxas por 100 mil habitantes** (fuzis apreendidos e mortes por intervenção policial). As taxas são muito assimétricas, então passam por `log1p` e por padronização (z-score) antes do K-Means. O número de grupos é fixado em 3 (baixa, intermediária e alta intensidade), e a célula de análise **testa essa escolha** (silhouette para k = 2 a 6) e a **estabilidade** da partição com 20 sementes. A associação entre as variáveis é medida por correlação de Spearman, no ano analisado e em cada ano da série.

In [ ]:
COLUNAS_CLUSTER = ["taxa_fuzil_100k", "taxa_intervencao_100k"]
K_CLUSTERS = 3
NOMES_NIVEIS = {3: ["Baixa intensidade", "Intensidade intermediária", "Alta intensidade (fuzis e intervenção)"]}
nomes_niveis = NOMES_NIVEIS.get(K_CLUSTERS, [f"Nível {i + 1}" for i in range(K_CLUSTERS)])
cores_niveis = [PALETA["azul"], PALETA["amarelo"], PALETA["vermelho"]]
if K_CLUSTERS != 3:
    cores_niveis = list(plt.cm.viridis(np.linspace(0, 1, K_CLUSTERS)))

# 1. Ano de análise: o mais recente com todas as variáveis disponíveis
candidatos = painel_aisp.dropna(subset=[*COLUNAS_CLUSTER, "populacao"])
ano_analise = int(candidatos["ano"].max())
df_ano = candidatos[candidatos["ano"] == ano_analise].copy()
n_painel_ano = int((painel_aisp["ano"] == ano_analise).sum())
print(
    f"Ano de análise: {ano_analise} (último com população). "
    f"AISPs no K-Means: {len(df_ano)}; fora por falta de dados: {n_painel_ano - len(df_ano)}."
)

# 2. Transformação e padronização
X_log = np.log1p(df_ano[COLUNAS_CLUSTER].to_numpy())
X_scaled = StandardScaler().fit_transform(X_log)

# 3. Escolha de k: silhouette e inércia para k = 2..6
diagnostico_k = []
for k in range(2, min(7, len(df_ano))):
    modelo_k = KMeans(n_clusters=k, random_state=SEMENTE, n_init=10).fit(X_scaled)
    diagnostico_k.append(
        {"k": k, "silhouette": silhouette_score(X_scaled, modelo_k.labels_), "inercia": modelo_k.inertia_}
    )
diagnostico_k = pd.DataFrame(diagnostico_k).round(3)
display(diagnostico_k)
melhor_k = int(diagnostico_k.loc[diagnostico_k["silhouette"].idxmax(), "k"])
print(f"Melhor silhouette: k = {melhor_k}. k usado na análise: {K_CLUSTERS} (definido pelos três níveis de intensidade).")

# 4. Modelo final e ordenação dos grupos pelo centroide (média dos dois eixos padronizados)
kmeans = KMeans(n_clusters=K_CLUSTERS, random_state=SEMENTE, n_init=10).fit(X_scaled)
ordem_grupos = np.argsort(kmeans.cluster_centers_.sum(axis=1))
nivel_do_grupo = {grupo: nivel for nivel, grupo in enumerate(ordem_grupos)}
df_ano["nivel"] = pd.Series(kmeans.labels_, index=df_ano.index).map(nivel_do_grupo)
df_ano["Classificação K-Means"] = df_ano["nivel"].map(dict(enumerate(nomes_niveis)))
df_ano["cor_cluster"] = df_ano["nivel"].map(dict(enumerate(cores_niveis)))
mapa_cores_nome = dict(zip(nomes_niveis, cores_niveis))

# 5. Estabilidade da partição frente à inicialização (ARI = 1 → partições idênticas)
aris = [
    adjusted_rand_score(kmeans.labels_, KMeans(n_clusters=K_CLUSTERS, random_state=s, n_init=1).fit_predict(X_scaled))
    for s in range(20)
]
print(f"Estabilidade (ARI contra o modelo final, 20 sementes): média = {np.mean(aris):.2f}, mínimo = {np.min(aris):.2f}")

# 6. Associação entre as duas taxas: no ano de análise e em cada ano da série
rho_ano, p_ano = stats.spearmanr(df_ano[COLUNAS_CLUSTER[0]], df_ano[COLUNAS_CLUSTER[1]])
print(f"Spearman entre fuzis e intervenção policial (por 100 mil), {ano_analise}: ρ = {rho_ano:.2f}, p = {p_ano:.4f} (n = {len(df_ano)})")

rho_por_ano = []
for ano, grupo in painel_aisp.dropna(subset=COLUNAS_CLUSTER).groupby("ano"):
    if len(grupo) >= 8:
        rho_a, p_a = stats.spearmanr(grupo[COLUNAS_CLUSTER[0]], grupo[COLUNAS_CLUSTER[1]])
        rho_por_ano.append({"ano": int(ano), "n_aisp": len(grupo), "rho_spearman": rho_a, "p_valor": p_a})
rho_por_ano = pd.DataFrame(rho_por_ano, columns=["ano", "n_aisp", "rho_spearman", "p_valor"])

print("\nAISPs por grupo:")
print(df_ano["Classificação K-Means"].value_counts().reindex(nomes_niveis).to_string())

# 7. Junção espacial (inner: só AISPs com dados e polígono; a perda é informada)
gdf_aisp_map = gdf_aisp.merge(df_ano, on="id_aisp", how="inner", validate="one_to_one")
if gdf_aisp_map.empty:
    raise ValueError(f"O cruzamento resultou em 0 linhas para o ano {ano_analise}. Verifique as chaves territoriais.")
print(f"\nAISPs com dados mas sem polígono (não aparecem nos mapas): {len(df_ano) - len(gdf_aisp_map)}.")


In [ ]:
fig = plt.figure(figsize=(22, 7))
grade = fig.add_gridspec(1, 3, width_ratios=[1.2, 1, 1], wspace=0.1)
ax_scatter = fig.add_subplot(grade[0])
ax_map1 = fig.add_subplot(grade[1])
ax_map2 = fig.add_subplot(grade[2])
x_col, y_col = COLUNAS_CLUSTER

# --- Dispersão com os grupos do K-Means ---
for nivel, nome in enumerate(nomes_niveis):
    subconjunto = df_ano[df_ano["nivel"] == nivel]
    ax_scatter.scatter(subconjunto[x_col], subconjunto[y_col], color=cores_niveis[nivel],
                       label=nome, s=75, alpha=0.9, edgecolors="black", linewidths=0.5)
ax_scatter.axvline(df_ano[x_col].median(), color="gray", linestyle="--", linewidth=1, alpha=0.6)
ax_scatter.axhline(df_ano[y_col].median(), color="gray", linestyle="--", linewidth=1, alpha=0.6,
                   label="Mediana entre AISPs (referência)")
destaques = pd.concat([df_ano.nlargest(3, c) for c in COLUNAS_CLUSTER]).drop_duplicates("id_aisp")
for _, linha in destaques.iterrows():
    ax_scatter.annotate(f"AISP {int(linha['id_aisp'])}", (linha[x_col], linha[y_col]),
                        xytext=(4, 4), textcoords="offset points", fontsize=8)
ax_scatter.set_title(f"K-Means (k = {K_CLUSTERS}) – {ano_analise}", fontsize=13, loc="left")
ax_scatter.set_xlabel("Fuzis apreendidos por 100 mil hab.")
ax_scatter.set_ylabel("Mortes por intervenção policial por 100 mil hab.")
ax_scatter.legend(frameon=True, loc="best", fontsize=9)
ax_scatter.spines[["top", "right"]].set_visible(False)

# --- Mapa 1: taxa de fuzis (AISPs sem dados em cinza claro) ---
gdf_aisp.plot(ax=ax_map1, color="#eeeeee", edgecolor="white", linewidth=0.3)
gdf_aisp_map.plot(column=x_col, cmap="Blues", linewidth=0.3, edgecolor="black", legend=True,
                  legend_kwds={"shrink": 0.5, "label": "Fuzis por 100 mil hab."}, ax=ax_map1)
ax_map1.set_title("Fuzis apreendidos (taxa)", fontsize=13)
ax_map1.axis("off")

# --- Mapa 2: grupos do K-Means ---
gdf_aisp.plot(ax=ax_map2, color="#eeeeee", edgecolor="white", linewidth=0.3)
gdf_aisp_map.plot(color=gdf_aisp_map["cor_cluster"], linewidth=0.3, edgecolor="black", ax=ax_map2)
ax_map2.legend(
    handles=[Patch(facecolor=cores_niveis[i], edgecolor="black", label=nomes_niveis[i]) for i in range(K_CLUSTERS)],
    loc="lower right", title="Grupo (K-Means)", frameon=True,
)
ax_map2.set_title("Distribuição territorial dos grupos", fontsize=13)
ax_map2.axis("off")

fig.suptitle(f"Fuzis apreendidos e mortes por intervenção policial nas AISPs do RJ ({ano_analise})",
             fontsize=16, fontweight="bold", y=1.02)
plt.show()

# --- Estabilidade temporal da associação ---
fig, ax = plt.subplots(figsize=(11, 4.5))
if rho_por_ano.empty:
    ax.text(0.5, 0.5, "Poucas AISPs por ano para estimar a associação", ha="center", transform=ax.transAxes)
else:
    ax.plot(rho_por_ano["ano"], rho_por_ano["rho_spearman"], marker="o", color=PALETA["azul"], linewidth=2)
    ax.set_xticks(rho_por_ano["ano"])
ax.axhline(0, color="gray", linewidth=1)
ax.set_ylim(-1, 1)
ax.tick_params(axis="x", rotation=45)
ax.set_ylabel("ρ de Spearman")
ax.set_title("A associação entre fuzis e intervenção policial (por 100 mil hab.) se mantém ao longo dos anos?", loc="left")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
gdf_idx = gdf_aisp_map.set_index("id_aisp")
dados_plot = gdf_idx.drop(columns="geometry")

fig_scatter = px.scatter(
    dados_plot.reset_index(),
    x=COLUNAS_CLUSTER[0],
    y=COLUNAS_CLUSTER[1],
    color="Classificação K-Means",
    color_discrete_map=mapa_cores_nome,
    category_orders={"Classificação K-Means": nomes_niveis},
    hover_name="rotulo_aisp",
    hover_data={
        COLUNAS_CLUSTER[0]: ":.2f", COLUNAS_CLUSTER[1]: ":.2f",
        "quantidade_fuzil": True, "Classificação K-Means": False,
    },
    title=f"Grupos de AISPs (K-Means) – {ano_analise}",
    labels={
        COLUNAS_CLUSTER[0]: "Fuzis apreendidos por 100 mil hab.",
        COLUNAS_CLUSTER[1]: "Mortes por intervenção policial por 100 mil hab.",
    },
)
fig_scatter.add_vline(x=df_ano[COLUNAS_CLUSTER[0]].median(), line_width=1, line_dash="dash", line_color="gray")
fig_scatter.add_hline(y=df_ano[COLUNAS_CLUSTER[1]].median(), line_width=1, line_dash="dash", line_color="gray")
fig_scatter.update_traces(marker=dict(size=12, line=dict(width=1, color="DarkSlateGrey")))
fig_scatter.update_layout(template="plotly_white", height=500)
fig_scatter.show()

fig_map = px.choropleth_map(
    dados_plot,
    geojson=gdf_idx.geometry,
    locations=dados_plot.index,
    color="Classificação K-Means",
    color_discrete_map=mapa_cores_nome,
    category_orders={"Classificação K-Means": nomes_niveis},
    hover_name="rotulo_aisp",
    hover_data={
        COLUNAS_CLUSTER[0]: ":.2f", COLUNAS_CLUSTER[1]: ":.2f",
        "quantidade_fuzil": True, "Classificação K-Means": False,
    },
    center={"lat": -22.55, "lon": -43.15},
    zoom=7.2,
    map_style="carto-positron",
    title=f"Distribuição territorial dos grupos (K-Means) – {ano_analise}",
)
fig_map.update_traces(marker_opacity=0.8, marker_line_width=0.5)
fig_map.update_layout(height=600, margin=dict(l=20, r=20, t=60, b=20))
fig_map.show()


### Leitura dos resultados e cuidados de interpretação

Os valores (ρ de Spearman, tamanho dos grupos, silhouette e estabilidade) são **impressos na célula de análise**, pois dependem do ano e dos dados carregados. A leitura correta combina três evidências: (1) o ρ do ano analisado; (2) o gráfico de ρ ao longo dos anos, que mostra se a relação é persistente ou pontual; (3) o mapa dos grupos, que mostra se a classificação tem padrão territorial.

**Os resultados descrevem associação, não causa.** Quatro cuidados:
- *Causalidade reversa e viés de medida:* apreensões de fuzis refletem a presença e a atividade policial no território; mortes por intervenção policial também. As duas variáveis podem crescer juntas porque ambas dependem da mesma atuação policial, e não porque um fenômeno cause o outro.
- *Falácia ecológica:* a análise é por AISP; nada pode ser inferido sobre indivíduos ou sobre áreas menores dentro de cada AISP.
- *Amostra pequena e um único ano:* são poucas dezenas de AISPs; por isso a partição é submetida ao teste de estabilidade e a relação é conferida ano a ano.
- *Grupos são descrição, não prova:* o K-Means sempre devolve k grupos, mesmo quando os dados não têm estrutura de grupos. A silhouette indica o quão separados eles são de fato.

### Fundamentação estatística: normalização, robustez e associação

- **Normalização.** Contagens de AISPs mais populosas são naturalmente maiores. Por isso as duas variáveis entram como **taxas por 100 mil habitantes**, $\frac{\text{ocorrências}}{\text{população}} \times 100.000$. (Na versão anterior, os fuzis entravam em valor absoluto e a intervenção policial em taxa, misturando duas escalas e dois critérios.)
- **Robustez.** Taxas territoriais são assimétricas, com alguns valores extremos. A transformação `log1p` e a padronização impedem que poucas AISPs dominem a distância euclidiana do K-Means. As **medianas** ($\tilde{x}$, $\tilde{y}$) aparecem apenas como referência visual (linhas tracejadas), pois são pouco sensíveis a outliers; **elas não definem os grupos**. Os grupos vêm exclusivamente do K-Means.
- **Associação.** Usa-se a correlação de **Spearman**, baseada em postos e, portanto, robusta a outliers e a relações monotônicas não lineares, acompanhada do teste de significância e da checagem ano a ano.

# Conclusões e limitações

Os resultados numéricos de cada pergunta são impressos nas células de **leitura dos resultados** (Perguntas 1, 2, 3 e 5), sempre a partir dos dados carregados, e não digitados à mão. Isso mantém o texto coerente caso as bases sejam atualizadas.

**Limitações que valem para todo o trabalho**
- *Denominador populacional:* para as AISPs, a população cobre só 2003–2022; os anos posteriores (inclusive os mais recentes das contagens) ficam sem taxa. Uma projeção populacional documentada permitiria estender as taxas, mas é uma decisão metodológica a ser tomada explicitamente.
- *Feminicídio:* a taxa usa a população total como denominador (a base não traz população feminina) e a série de 2016 foi excluída por registro parcial.
- *Mudanças territoriais:* a relação CISP–AISP é tratada como fixa; reorganizações históricas de áreas podem afetar comparações entre anos. O número de AISPs com taxa por ano é mostrado na Pergunta 2 para dar visibilidade a esse risco.
- *Registro administrativo:* os dados vêm de registros policiais. Variações de registro (como a melhora na tipificação de feminicídio) podem parecer variações reais.
- *Correlações entre séries anuais:* poucos pontos e tendências comuns; interpretadas apenas como descrição.
- *Perguntas 4, 6 e 7:* não fazem parte deste notebook.